<a href="https://colab.research.google.com/github/Amazeble/chatterbox-finetuning/blob/main/chatterbox_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Chatterbox TTS Fine-Tuning on Google Colab

This notebook provides a complete workflow for fine-tuning Chatterbox TTS models (Standard and Turbo) with LoRA support.

## Overview
- **Step 1**: Clone repository
- **Step 2**: Connect Google Drive and Copy Dataset
- **Step 3**: Install Dependencies
- **Step 4**: Download & Prepare Models (setup.py)
- **Step 5**: Configure Training Parameters
- **Step 6**: Preprocess Dataset
- **Step 7**: Train the Model
- **Step 8**: Inference (Generate Speech)
- **Step 9**: Merge LoRA Adapter (Optional)

### ⚠️ Important Notes
- **GPU Runtime Required**: Go to `Runtime > Change runtime type > GPU`
- **LoRA Recommended**: For datasets < 10 hours, use LoRA (`is_lora = True`)
- **Turbo Mode**: Faster training but may require LoRA for stability


## Step 1: Clone Repository

In [ ]:
#@title 📥 Clone Chatterbox Repository
#@markdown This cell clones the Chatterbox TTS repository from GitHub.

import os

# Detect environment and set working directory
if os.path.exists('/content'):
    # Running in Google Colab
    WORKSPACE = '/content'
    %cd /content
    print("Detected Google Colab environment")
else:
    # Running locally or in other environments
    WORKSPACE = '/workspace'
    %cd "$WORKSPACE"
    print("Running in local/other environment")

# Clone the repository if it doesn't exist
if not os.path.exists('chatterbox-finetuning'):
    print("Cloning Chatterbox Finetuning repository...")
    !git clone https://github.com/Amazeble/chatterbox-finetuning.git chatterbox
    %cd chatterbox
    print("✅ Repository cloned successfully!")
else:
    print("✅ Repository already exists.")
    %cd chatterbox

# Verify contents
print("\nRepository contents:")
!ls -la

## Step 2: Connect Google Drive and Copy Dataset

In [ ]:
#@title 📁 Connect Google Drive and Copy Dataset
#@markdown This cell mounts Google Drive and copies your dataset to the working directory.

import os
from google.colab import drive

# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Define source and destination paths
source_path = '/content/drive/MyDrive/MyTTSDataset/drive/my-drive'
dest_path = '/content/chatterbox/'

# Check if source path exists
if os.path.exists(source_path):
    print(f"\n✅ Source path found: {source_path}")
    print("Copying dataset to working directory...")
    
    # Create destination directory if it doesn't exist
    os.makedirs(dest_path, exist_ok=True)
    
    # Copy files
    !cp -r {source_path}/* {dest_path}
    
    print(f"✅ Dataset copied successfully to {dest_path}")
    print("\nVerifying copied files:")
    !ls -la {dest_path}
else:
    print(f"⚠️ Source path not found: {source_path}")
    print("Please ensure your dataset is in the correct location in Google Drive.")

## Step 3: Install Dependencies

In [ ]:
#@title 🔧 Install Dependencies
#@markdown This cell installs all required Python packages and FFmpeg.

import os

# Install FFmpeg
print("Installing FFmpeg...")
!apt-get update && apt-get install -y ffmpeg

# Install Python dependencies
print("\nInstalling Python dependencies...")
%cd /content/chatterbox
!pip install -r requirements.txt

print("\n✅ Dependencies installed successfully!")

## Step 4: Download & Prepare Models

In [ ]:
#@title 📥 Download & Prepare Models (setup.py)
#@markdown This cell runs setup.py to download base models and prepare tokenizers.
#@markdown **IMPORTANT**: After running this, check the output for the vocab_size value and update it in Step 5!

%cd /content/chatterbox

# Run setup.py
print("Running setup.py to download models and prepare tokenizers...")
!python setup.py

print("\n✅ Setup completed!")
print("⚠️ IMPORTANT: If you're using Turbo mode, copy the vocab_size value from the output above!")
print("You'll need to update it in the config file in Step 5.")

## Step 5: Configure Training Parameters

In [ ]:
#@title ⚙️ Configure Training Parameters
#@markdown Edit the src/config.py file to set your training preferences.

%cd /content/chatterbox

# Display current config
print("Current configuration (src/config.py):")
print("="*50)
!cat src/config.py

print("\n" + "="*50)
print("To modify settings, edit the values in src/config.py")
print("Key settings to check:")
print("  - is_turbo: True/False (Turbo mode)")
print("  - is_lora: True/False (LoRA training - recommended for <10h data)")
print("  - new_vocab_size: Must match tokenizer.json (update after setup.py if Turbo)")
print("  - ljspeech/json_format: Dataset format")
print("  - preprocess: Set to True for first run")

# Uncomment and modify the following to update config automatically
# !sed -i 's/is_turbo: bool = False/is_turbo: bool = True/' src/config.py
# !sed -i 's/is_lora: bool = False/is_lora: bool = True/' src/config.py
# !sed -i 's/new_vocab_size: int = .*/new_vocab_size: int = 52260/' src/config.py  # Update with value from setup.py

print("\n✅ Review the config above and modify as needed!")

## Step 6: Preprocess Dataset

In [ ]:
#@title 🔄 Preprocess Dataset
#@markdown This cell preprocesses your dataset for training. Make sure preprocess=True in config.py first!

%cd /content/chatterbox

print("Preprocessing dataset...")
print("This may take several minutes depending on your dataset size.")

# Run preprocessing by executing train.py with preprocess=True
# The preprocessing happens automatically when preprocess=True in config.py
!python train.py

print("\n✅ Preprocessing completed!")
print("If you want to train without preprocessing next time, set preprocess=False in config.py")

## Step 7: Train the Model

In [ ]:
#@title 🏋️ Train the Model
#@markdown Start the fine-tuning process. This will take several hours depending on your dataset and GPU.

%cd /content/chatterbox

print("Starting training...")
print("Training progress and audio samples will be displayed below.")
print("Press Ctrl+C to stop training early.")

# Start training
!python train.py

print("\n✅ Training completed!")
print("Check the chatterbox_output/ directory for your trained model.")

## Step 8: Inference (Generate Speech)

In [ ]:
#@title 🗣️ Inference - Generate Speech
#@markdown Test your trained model by generating speech from text.

%cd /content/chatterbox

# First, let's check what models are available
print("Checking available models...")
!ls -la chatterbox_output/

print("\n" + "="*50)
print("To generate speech, edit inference.py with your desired text:")
print("  TEXT_TO_SAY = \"Your text here\"")
print("  AUDIO_PROMPT = \"./speaker_reference/reference.wav\"")
print("="*50)

# Run inference
print("\nRunning inference...")
!python inference.py

print("\n✅ Speech generated!")
print("Output saved as output.wav")

# Play the generated audio
from IPython.display import Audio, display
print("\nPlaying generated audio:")
display(Audio('./output.wav'))

## Step 9: Merge LoRA Adapter (Optional)

In [ ]:
#@title 📦 Merge LoRA Adapter (Optional)
#@markdown If you trained with LoRA, this cell merges the adapter into a single standalone model file.

%cd /content/chatterbox

print("Merging LoRA adapter into base model...")
print("This creates a standalone .safetensors file that doesn't require LoRA adapters.")

!python merge_lora.py

print("\n✅ Merge completed!")
print("Your merged model is ready for production use.")
print("Check chatterbox_output/ for the merged .safetensors file.")